In [3]:
# %pip install -e ../../wazeasy
# %pip install folium mapclassify matplotlib

In [1]:
from wazeasy import utils
import dask.dataframe as dd
import pandas as pd
import numpy as np
import yaml

import geopandas as gpd
from shapely import wkt

In [2]:
with open("../docs/config_montevideo.yaml", "r") as f:
    config = yaml.safe_load(f)

timezone = config["Montevideo"]["timezone"]["timezone_name"]
s3_path = config["Montevideo"]["s3_path"]
aws_profile = config["Montevideo"]["aws_profile"]
geographies = config["geographies"]

In [3]:
ddf = dd.read_parquet(s3_path + '*parquet', storage_options={'profile': aws_profile}, engine='pyarrow')
utils.handle_time(ddf, timezone)

In [4]:
start_date = "2026-04-01"
end_date =  "2026-04-30"
name = start_date[:7].replace('-', '')
dates_of_interest = utils.define_dates_of_interest(ddf, start_date, end_date)
df_filt = ddf[ddf["date"].isin(dates_of_interest)]
df_filt = utils.assign_geography_to_jams(df_filt, geographies)
df_filt = df_filt.compute()
df_filt.to_parquet("s3://wbg-waze/bq/UY/784montevideo/jams/month_of_interest/{}-jams.parquet".format(name), 
                   engine = 'pyarrow',
                   index = False,
                   compression = 'snappy',
                   storage_options={"profile": "ddp"})